In [1]:
%load_ext autoreload
%autoreload 2

from scaling_llms.experiments import build_trainer
from scaling_llms.trainer import TrainerConfig
from scaling_llms.models import GPTConfig, GPTModel
from scaling_llms.utils.io import load_module_from_path, get_local_repo_dir
from scaling_llms.data import get_vocab_size
import gc
import torch
from torch.utils.data import IterableDataset, DataLoader


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class RandomTokenDataset(IterableDataset):
    """
    Infinite stream of random token sequences.
    Useful for synthetic benchmarking / coord checks.
    """

    def __init__(self, vocab_size: int, seq_len: int, seed: int = 42):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.seed = seed

    def __iter__(self):
        g = torch.Generator()
        g.manual_seed(self.seed)

        while True:
            x = torch.randint(
                0,
                self.vocab_size,
                (self.seq_len,),
                generator=g,
            )
            y = torch.randint(
                0,
                self.vocab_size,
                (self.seq_len,),
                generator=g,
            )
            yield x, y


def make_synthetic_dataloader(
    batch_size: int,
    seq_len: int,
    vocab_size: int,
    seed: int = 42,
    num_workers: int = 0,
) -> DataLoader:
    """
    Returns an infinite DataLoader of random token batches.

    Output:
        x: (B, T)
        y: (B, T)
    """

    dataset = RandomTokenDataset(
        vocab_size=vocab_size,
        seq_len=seq_len,
        seed=seed,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
    )

In [ ]:
def fits_batch_size(
    batch_size: int,
    seq_len: int,
    vocab_size: int,
    width: int,
    gpt_hparams: dict,
    device: str = "cuda",
):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    trainer = None
    raw_model = None
    train_dl = None

    try:
        trainer_cfg = TrainerConfig(
            num_steps=1,
            accum_steps=1,
            precision="bf16",
            device=device,
            use_compile=False,
        )

        gpt_cfg = GPTConfig(
            n_embd=width,
            seq_len=seq_len,
            vocab_size=vocab_size,
            **gpt_hparams,
        )

        raw_model = GPTModel(gpt_cfg)

        train_dl = make_synthetic_dataloader(
            batch_size=batch_size,
            seq_len=seq_len,
            vocab_size=vocab_size,
        )

        trainer = build_trainer(
            cfg=trainer_cfg,
            raw_model=raw_model,
            train_dl=train_dl,
            eval_dl=None,
            run=None,
        )

        trainer.train(max_steps=1)

        peak_gb = torch.cuda.max_memory_allocated() / 1024**3
        return True, peak_gb

    except torch.cuda.OutOfMemoryError:
        return False, None

    finally:
        del trainer, raw_model, train_dl
        gc.collect()
        torch.cuda.empty_cache()


In [6]:
def scan_batch_sizes_pow2(
    seq_len: int,
    vocab_size: int,
    width: int,
    gpt_hparams: dict,
    start_bs: int = 1,
    max_power: int = 16,
    device: str = "cuda",
):
    assert start_bs > 0 and (start_bs & (start_bs - 1)) == 0, "start_bs must be a power of 2"

    results = {}

    bs = start_bs
    for _ in range(max_power):
        ok, peak_gb = fits_batch_size(
            batch_size=bs,
            seq_len=seq_len,
            vocab_size=vocab_size,
            width=width,
            gpt_hparams=gpt_hparams,
            device=device,
        )

        results[bs] = {
            "fits": ok,
            "peak_gb": peak_gb,
        }

        if not ok:
            break

        bs *= 2

    return results

In [4]:
CONFIGS_ROOT = get_local_repo_dir() / "configs"
EXP_CONFIGS_ROOT = CONFIGS_ROOT / "experiments" /"mup_test"
RUNTIME_CONFIGS_ROOT = CONFIGS_ROOT / "runtime"
configs = load_module_from_path(EXP_CONFIGS_ROOT / "training_curves.py")

In [7]:
results = scan_batch_sizes_pow2(
    seq_len=configs.DATALOADER_KWARGS["seq_len"],
    vocab_size=get_vocab_size(configs.DATASET_KWARGS["tokenizer_name"]),
    width=max(configs.WIDTHS),
    gpt_hparams=configs.CONSTANT_GPT_HPARAMS,
    start_bs=8,
)

print(results)

2026-04-19 17:09:05 | Trainer | Starting Training
2026-04-19 17:09:05 | Trainer | [model] params=178,437,120 | n_layer=6 | n_embd=1024 | vocab_size=50,257
2026-04-19 17:09:05 | Trainer | [device] device=cuda:0 | device_name=nvidia a100-sxm4-80gb | world_size=1 | precision=bf16
2026-04-19 17:09:05 | Trainer | [optimization] max_num_steps=1 | remaining_steps=1 | accum_steps=1 | lr=3.000e-04 | warmup_steps=0 | lr_schedule=none
2026-04-19 17:09:05 | Trainer | [train] step=0 | tokens_total=8,192 | nll=11.3173 | ppl=82232.87 | lr=3.000e-04 | tok_per_s=139566 | step_ms=58.7


/workspace/repos/scaling-llms/src/scaling_llms/experiments.py:81: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(


2026-04-19 17:09:13 | Trainer | Starting Training
2026-04-19 17:09:13 | Trainer | [model] params=178,437,120 | n_layer=6 | n_embd=1024 | vocab_size=50,257
2026-04-19 17:09:13 | Trainer | [device] device=cuda:0 | device_name=nvidia a100-sxm4-80gb | world_size=1 | precision=bf16
2026-04-19 17:09:13 | Trainer | [optimization] max_num_steps=1 | remaining_steps=1 | accum_steps=1 | lr=3.000e-04 | warmup_steps=0 | lr_schedule=none
2026-04-19 17:09:13 | Trainer | [train] step=0 | tokens_total=16,384 | nll=11.3195 | ppl=82410.45 | lr=3.000e-04 | tok_per_s=321547 | step_ms=51.0
2026-04-19 17:09:20 | Trainer | Starting Training
2026-04-19 17:09:20 | Trainer | [model] params=178,437,120 | n_layer=6 | n_embd=1024 | vocab_size=50,257
2026-04-19 17:09:20 | Trainer | [device] device=cuda:0 | device_name=nvidia a100-sxm4-80gb | world_size=1 | precision=bf16
2026-04-19 17:09:20 | Trainer | [optimization] max_num_steps=1 | remaining_steps=1 | accum_steps=1 | lr=3.000e-04 | warmup_steps=0 | lr_schedule=no

In [8]:
results

{8: {'fits': True, 'peak_gb': 7.913272857666016},
 16: {'fits': True, 'peak_gb': 14.84421157836914},
 32: {'fits': True, 'peak_gb': 28.70529556274414},
 64: {'fits': True, 'peak_gb': 56.42563247680664},
 128: {'fits': False, 'peak_gb': None}}